# Setup
## Load Packages

In [1]:
library(bayestestR)
library(dplyr)
library(tidyr)


Attaching package: ‘dplyr’


The following objects are masked from ‘package:stats’:

    filter, lag


The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union




SaSETU already registered with postscriptFont().

SbSETU already registered with postscriptFont().

No regular (non-bold, non-italic) version of ScSETU. Skipping setup for this font.

STIX Two Text already registered with postscriptFont().

System Font already registered with postscriptFont().

Tahoma already registered with postscriptFont().

Times New Roman already registered with postscriptFont().

Trattatello already registered with postscriptFont().

Trebuchet MS already registered with postscriptFont().

Verdana already registered with postscriptFont().

Webdings already registered with postscriptFont().

Wingdings already registered with postscriptFont().

Wingdings 2 already registered with postscriptFont().

Wingdings 3 already registered with postscriptFont().



## Import and Wrangle data

In [2]:
file_paths = list.files(path = ".", pattern="\\.csv$", full.names = T)
file_paths
files = lapply(file_paths, read.csv)
tail(files)

[1] "./FA1.csv"  "./FA10.csv" "./FA2.csv"  "./FA3.csv"  "./FA4.csv" 
 [6] "./FA5.csv"  "./FA6.csv"  "./FA7.csv"  "./FA8.csv"  "./FA9.csv"

Date,FA.,Condition,Temperature,Colony,Rep_1,Rep_2,Rep_3,Rep_4,Rep_5
<chr>,<dbl>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
2024-02-12 10:00:00,4,20S,37,1,0.081,0.073,0.078,0.099,0.087
2024-02-12 10:00:00,4,20S,20,1,0.070,0.075,0.093,0.076,0.082
2024-02-12 10:00:00,4,20S,37,2,0.073,0.094,0.089,0.102,0.095
2024-02-12 10:00:00,4,20S,20,2,0.057,0.081,0.063,0.087,0.076
2024-02-12 10:00:00,4,20S,37,3,0.067,0.078,0.075,0.058,0.083
2024-02-12 10:00:00,4,20S,20,3,0.075,0.073,0.071,0.067,0.069
2024-02-12 10:00:00,4,20S,37,4,0.078,0.088,0.076,0.079,0.090
2024-02-12 10:00:00,4,20S,20,4,0.077,0.088,0.062,0.080,0.066
2024-02-12 10:00:00,4,20S,37,5,0.064,0.105,0.062,0.073,0.068


In [3]:
# Convert the date and time and fix the column names for each of the dataframes
for (x in 1:length(file_paths)) {
    files[[x]]$Date <- as.POSIXct(files[[x]]$Date)
    files[[x]]$Date <- as.numeric(difftime(files[[x]]$Date, min(files[[x]]$Date), units = "hours"))
    names(files[[x]])[names(files[[x]]) == 'FA.'] <- 'FA'
    names(files[[x]])[names(files[[x]]) == 'Date'] <- 'Hours'
}

In [4]:
#combine into one dataframe
data <- do.call("rbind", files) 
head(data)
dim(data)

,Hours,FA,Condition,Temperature,Colony,Rep_1,Rep_2,Rep_3,Rep_4,Rep_5
,<dbl>,<dbl>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
1,56,1,A,37,1,1.144,1.087,1.073,1.187,1.107
2,56,1,A,20,1,1.446,1.574,1.379,1.479,1.410
3,56,1,A,37,2,0.730,0.914,0.796,0.832,0.776
4,56,1,A,20,2,1.197,1.190,1.512,1.366,0.872
5,56,1,A,37,3,0.870,0.727,0.699,0.748,0.648
6,56,1,A,20,3,1.298,1.517,1.491,1.473,1.481


[1] 7584   10

In [5]:
# Condense the Reps into one column and combine columns to make it easier to process
temp <- data %>%
            pivot_longer(cols = starts_with("Rep_"), 
                         names_to = "Rep", 
                         names_prefix = "Rep_", 
                         values_to = "OD") %>%
            unite(group, c("FA", "Condition", "Temperature", "Colony", "Rep"))
temp$Area <- 0
head(temp)
dim(temp)

Hours,group,OD,Area
<dbl>,<chr>,<dbl>,<dbl>
56,1_A_37_1_1,1.144,0
56,1_A_37_1_2,1.087,0
56,1_A_37_1_3,1.073,0
56,1_A_37_1_4,1.187,0
56,1_A_37_1_5,1.107,0
56,1_A_20_1_1,1.446,0


[1] 37920     4

# Calculate AUC
Opted for the cumulative AUC, wherein the area of the current time segment is added to the previous segment area. This method was chosen as the more logical option, compared to calculating the AUC at each timepoint from t0. Both methods have a correlation of 0.99.  

In [6]:
groups <- unique(temp$group)
length(groups)

[1] 4080

In [7]:
area_temp <- c()
groups <- unique(temp$group)

for (i in 1:length(groups)){
    #subset the group and put in order
    subset_df <- arrange(temp[temp$group == groups[i],], Hours)
    
    times <- sort(unique(subset_df$Hours))
    previous_section_area <- 0
    for (t in 2:length(times)){
        #subset for the current time and the previous time
        subset_time_df <- subset_df %>% filter(Hours == times[t] | Hours == times[t-1])
        
        #Calculate current section area
        current_section_area <- area_under_curve(subset_time_df$Hours, subset_time_df$OD, method = "spline")
        
        # Add it to the previous section area
        subset_df[subset_df$Hours == times[t], "Area"] <- current_section_area + previous_section_area
        
        # Overwrite the previous section area to the current section area
        previous_section_area <- current_section_area + previous_section_area
    }
    area_temp[[i]] <- data.frame(subset_df)
}

## Clean up

In [8]:
area_df <- do.call("rbind", area_temp) %>% 
            separate(group, c("FA", "Condition", "Temperature", "Colony", "Rep")) %>%
            mutate_at(c("Temperature", "Rep"), factor) %>%
            mutate_at(c("FA", "Colony"), as.numeric)
head(area_df)
dim(area_df)

,Hours,FA,Condition,Temperature,Colony,Rep,OD,Area
,<dbl>,<dbl>,<chr>,<fct>,<dbl>,<fct>,<dbl>,<dbl>
1,0,1,A,37,1,1,0.074,0.000
2,8,1,A,37,1,1,0.407,1.924
3,24,1,A,37,1,1,1.296,15.548
4,32,1,A,37,1,1,1.194,25.508
5,48,1,A,37,1,1,1.151,44.268
6,56,1,A,37,1,1,1.144,53.448


[1] 37920     8

In [9]:
#Create copies of the original data for each condition for t0
area_df_A <- area_df[area_df$Condition == "A",]
dim(area_df_A)
area_df_A$Condition <- "37S"
area_df_S <- area_df_A
area_df_S$Condition <- "20S"
temp <- rbind(area_df_A, area_df_S)
area_df_S$Condition <- "P"
temp <- rbind(temp, area_df_S)
area_df_noA <- rbind(temp, area_df[area_df$Condition != "A",])
table(area_df_noA$Condition)

[1] 1200    8


  20S   25S   37S     P 
11280  6480 11280 11280 

In [10]:
area_df_noA$Hours <- area_df_noA$Hours / 24
names(area_df_noA)[names(area_df_noA) == 'Hours'] <- 'Days'
condition_names <- c("20S" = "20°C Specialist",
                    "37S" = "37°C Specialist",
                    "P" = "Plastic/Fluctuating", 
                    "25S" = "25°C Control")

## Export

In [12]:
write.csv(area_df_noA, "../HE_area_table_additive.csv", row.names = F)